In [2]:
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()
client = OpenAI()

response = client.responses.create(
    model="gpt-4.1-mini",
    input="Buat 3 insight bisnis dari data penjualan skincare."
)

print(response.output_text)


Tentu! Berikut adalah contoh 3 insight bisnis yang bisa diambil dari data penjualan skincare:

1. **Produk Favorit dan Tren Konsumen**  
   Dengan menganalisis produk skincare mana yang paling banyak terjual, bisnis dapat mengidentifikasi produk favorit konsumen. Misalnya, jika serum vitamin C atau pelembap dengan SPF memiliki penjualan yang tinggi, maka tren perawatan kulit yang mengutamakan pencerahan dan perlindungan dari sinar matahari sedang naik daun. Ini membantu dalam mengembangkan dan mempromosikan produk serupa atau varian yang lebih inovatif.

2. **Segmen Pasar Berdasarkan Demografi dan Preferensi**  
   Data penjualan yang ditambahkan dengan informasi demografis seperti usia, jenis kelamin, dan lokasi pelanggan bisa memberikan insight tentang siapa target utama produk skincare. Misalnya, jika produk anti-aging lebih banyak dibeli oleh pelanggan berusia 35 tahun ke atas, sedangkan produk pembersih wajah lebih diminati oleh usia 20-30 tahun, maka strategi pemasaran dan penawa

In [3]:
import sqlite3
import pandas as pd

orders = pd.DataFrame([
  {"date":"2026-05-01", "channel":"TikTok", "sku":"SSUP", "qty":45, "revenue":10080000, "region":"Jawa Barat"},
  {"date":"2026-05-01", "channel":"Shopee", "sku":"ADC",  "qty":30, "revenue":6720000,  "region":"DKI Jakarta"},
  {"date":"2026-05-02", "channel":"TikTok", "sku":"TEF",  "qty":55, "revenue":8250000,  "region":"Jawa Timur"},
  {"date":"2026-05-02", "channel":"Shopee", "sku":"ECG",  "qty":18, "revenue":3150000,  "region":"Banten"}
])

con = sqlite3.connect("../data/retail_bi.db")
orders.to_sql("orders", con, if_exists="replace", index=False)
con.close()


In [6]:
import sqlite3
import pandas as pd

with sqlite3.connect("../data/retail_bi.db") as con:
    query = """
    SELECT
        channel,
        sku,
        SUM(qty) AS total_qty,
        SUM(revenue) AS total_revenue
    FROM orders
    GROUP BY channel, sku
    ORDER BY total_revenue DESC;
"""
    df = pd.read_sql_query(query, con)
df.head()

df


,channel,sku,total_qty,total_revenue
0,TikTok,SSUP,45,10080000
1,TikTok,TEF,55,8250000
2,Shopee,ADC,30,6720000
3,Shopee,ECG,18,3150000


In [8]:
summary_table = df.to_markdown(index=False)

prompt = f"""
Anda adalah BI Analyst untuk brand skincare.
Gunakan data penjualan berikut:

{summary_table}

Tugas:
1. Tulis 3 insight utama.
2. Jelaskan kemungkinan penyebab bisnis.
3. Berikan 3 rekomendasi aksi.
4. Jangan mengarang angka di luar tabel.
"""

response = client.responses.create(
    model="gpt-4o-mini",
    input=prompt
)

print(response.output_text)


### 1. Tiga Insight Utama
- **Dominasi Penjualan TikTok**: Channel TikTok menunjukkan performa yang jauh lebih baik dibandingkan Shopee, dengan total penjualan (total_qty) mencapai 100 unit, berkontribusi pada revenue sebesar 18.840.000.000, yang hampir dua kali lipat dari Shopee yang hanya mencapai total revenue 9.870.000.
  
- **Kinerja SKU Antara Channel**: SKU SSUP di TikTok berhasil menjual 45 unit, menjadikannya sebagai produk unggulan. Sebaliknya, di Shopee, meskipun dua SKU tercatat, jumlah total penjualannya lebih rendah (48 unit) dengan revenue yang lebih rendah pula.

- **Efektivitas SKU di Channel Berbeda**: SKU TEF di TikTok terjual 55 unit dengan revenue 8.250.000, sementara SKU yang sama tidak tercatat di Shopee, menunjukkan bahwa TikTok mungkin lebih cocok untuk kategori produk tertentu, sedangkan Shopee belum berhasil menarik minat pelanggan terhadap produk tersebut.

---

### 2. Kemungkinan Penyebab Bisnis
- **Target Pasar yang Berbeda**: TikTok mungkin memiliki audie

In [11]:
def generate_business_insight(df, question):
    table = df.to_markdown(index=False)
    prompt = f"""
    Anda adalah BI Analyst.
    Jawab pertanyaan bisnis berikut: {question}

    Data:
    {table}

    Format output:
    - Executive summary
    - 3 insight utama
    - Risiko / asumsi data
    - Rekomendasi aksi
    """
    res = client.responses.create(
        model="gpt-4o-mini",
        input=prompt
    )
    return res.output_text

insight = generate_business_insight(
    df,
    "SKU dan channel mana yang paling berkontribusi pada revenue?"
)
print(insight)


### Executive Summary
Analisis ini bertujuan untuk mengidentifikasi SKU dan saluran penjualan (channel) yang paling berkontribusi terhadap revenue. Berdasarkan data yang tersedia, channel TikTok dan SKU SSUP menunjukkan kontribusi revenue yang paling tinggi, yang menegaskan peran penting dari platform ini dan produk tertentu dalam strategi penjualan.

### 3 Insight Utama
1. **Kontribusi Revenue Tertinggi:**
   - SKU **SSUP** pada channel **TikTok** adalah produk yang memberikan kontribusi revenue tertinggi sebesar **10.080.000** IDR dengan jumlah penjualan **45 unit**, menandakan popularitas dan keberhasilan pemasaran produk ini di platform tersebut.

2. **Perbandingan Channel:**
   - Channel **TikTok** secara keseluruhan menghasilkan **18.825.000 IDR** dari dua SKU (SSUP dan TEF), sedangkan channel **Shopee** hanya menghasilkan **9.975.000 IDR** dari dua SKU (ADC dan ECG). Ini menunjukkan bahwa TikTok lebih efektif dalam menghasilkan revenue dibandingkan Shopee.

3. **Efektivitas SKU 

In [12]:
from pathlib import Path

Path("outputs").mkdir(exist_ok=True)

with open("../outputs/insight_sample.md", "w", encoding="utf-8") as f:
    f.write("# AI Business Insight - Sesi 1\n\n")
    f.write(insight)

print("Report saved to outputs/insight_sample.md")


Report saved to outputs/insight_sample.md
